# Segment 2: Tool Design, Integration, and Claude Code Workflows

**Duration:** 50 minutes
**Maps to:** CCA-F Domain 2 (Tools/MCP, 18%) + Domain 3 (Claude Code, 20%) = **38% of the exam**
**References:** [`../domain-2-tools-mcp.md`](../domain-2-tools-mcp.md), [`../domain-3-claude-code.md`](../domain-3-claude-code.md)

In Segment 1 you wrote the agent. Now we go one level deeper: the **tools** the agent calls, the **MCP servers** that ship those tools to multiple agents, and the **Claude Code instruction hierarchy** that lets a team enforce conventions without writing them in every prompt.


## Learning objectives

- Write **tool definitions** whose `description` and `input_schema` carry the contract, not the name
- Use **`tool_choice`** modes (`auto`, `any`, `tool`, `none`) to constrain agent behavior
- Return **structured tool errors** so the model can decide to retry, reformulate, or escalate
- Configure **`.mcp.json`** for stdio, SSE, and HTTP transports with `${ENV_VAR}` expansion
- Reason about the **CLAUDE.md hierarchy** (user, project, subtree, local) and its precedence
- Run Claude Code **non-interactively** with `claude -p` for CI/CD


## The description is the contract

The Messages API tool shape is `name`, `description`, `input_schema`, and optional `cache_control`. Names are labels. **The description is the contract.** Spell out:

- What the tool does
- When the agent should call it
- When the agent should **not** call it
- What each input means
- What the success and failure shapes look like

Thin descriptions ("gets weather") leak responsibility into the prompt and the agent's improvisation. Opinionated descriptions let you delete prompt instructions you used to need.


## The four `tool_choice` modes

| Mode | Shape | When to use |
|---|---|---|
| `auto` | `{"type": "auto"}` | Default. Model decides. Supports `disable_parallel_tool_use`. |
| `any` | `{"type": "any"}` | Force the model to call *some* tool this turn. Useful for classification gates. |
| `tool` | `{"type": "tool", "name": "<tool_name>"}` | Force a *specific* tool. This is the lever for **structured output** in Segment 3. |
| `none` | `{"type": "none"}` | No tools this turn. The model must answer in prose. |

Set `disable_parallel_tool_use: true` when one tool's output feeds the next.


## Structured tool errors

When a tool returns an error, return it **as a structured object**, not a string:

```json
{
  "isError": true,
  "errorCategory": "transient | permanent | policy",
  "isRetryable": true,
  "message": "..."
}
```

The model reads `errorCategory` and `isRetryable` to decide. Categories:

- **`transient`** - retry with backoff (network blip, rate limit)
- **`permanent`** - do not retry; reformulate (bad input, unknown ID)
- **`policy`** - do not retry; escalate (over-cap refund, unauthorized action)

This is the same shape Segment 1's `enforce_refund_policy` hook returned. Same pattern, different surface.


## MCP: three transports, one variable-expansion rule

**Model Context Protocol** (MCP) is how you ship tools to *multiple* agents from a single server. The client config is just JSON:

| Transport | Use case | Shape |
|---|---|---|
| **stdio** | Local servers, scripts | `{"command": "npx", "args": [...], "env": {"VAR": "${VAR}"}}` |
| **SSE** | Streaming events, server push | `{"type": "sse", "url": "https://...", "headers": {...}}` |
| **HTTP** | Request/response, simple integration | `{"type": "http", "url": "...", "headers": {...}}` |

**`${ENV_VAR}` expansion** works in `env`, `args`, and `headers`. This is how secrets stay out of the file. **Never** commit a literal token; always use `${GITHUB_TOKEN}` (or similar) and source it from the shell.

The file goes at `.mcp.json` (project) or `~/.claude.json` (user). Project values override user values at the server level.


## The CLAUDE.md instruction hierarchy

Four tiers, in precedence order from broadest to most specific:

| Tier | Path | Purpose |
|---|---|---|
| **User** | `~/.claude/CLAUDE.md` | Your personal defaults, every project |
| **Project** | `<repo>/CLAUDE.md` | Team conventions, checked in, version-controlled |
| **Subtree** | `<repo>/<subdir>/CLAUDE.md` | Loaded on demand when files in that subtree are touched |
| **Local override** | `<repo>/CLAUDE.local.md` | Gitignored, personal tweaks for one machine |

The Agent SDK loads user + project when you pass `settingSources=["user", "project"]`.

**Subtree files are the unsung hero.** A frontend rule does not need to pollute a backend file. Put React conventions in `frontend/CLAUDE.md`; they only load when you read frontend files.


## Demo: tool definitions, tool_choice modes, and the structured error contract

We will exercise each concept above with a small live call. Keep an eye on:

- How the same tool behaves differently with a **thin** vs **opinionated** description
- How `stop_reason` changes when you flip `tool_choice` modes
- How the model reacts to a `policy` error vs a `transient` error
- How `.mcp.json` reads as plain JSON config (no MCP client invocation here, just config-as-data)


In [ ]:
import json
import os
from pathlib import Path

from anthropic import Anthropic

try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent / ".env")
except ImportError:
    pass

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
client = Anthropic()
MODEL = "claude-sonnet-4-6"


## Thin vs opinionated descriptions

Same tool name, two descriptions. Same user prompt. Watch how the second version changes the model's behavior **without changing the prompt**.


In [ ]:
THIN_WEATHER = {
    "name": "get_weather",
    "description": "gets weather",
    "input_schema": {
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"],
    },
}

OPINIONATED_WEATHER = {
    "name": "get_weather",
    "description": (
        "Get the current weather conditions for a specific city. Returns "
        "temperature in Celsius, conditions (clear, cloudy, rain, snow), and "
        "humidity percent. Call this when the user asks about weather. Do NOT "
        "call for forecasts more than 24 hours out - this tool only returns "
        "current conditions. If the city is ambiguous (e.g. 'Springfield'), "
        "ask the user to disambiguate BEFORE calling."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name; disambiguate with region if ambiguous"},
            "region": {"type": "string", "description": "Optional state/country to disambiguate"},
        },
        "required": ["city"],
    },
}

PROMPT = "What's the weather in Springfield?"

for label, tool in [("thin", THIN_WEATHER), ("opinionated", OPINIONATED_WEATHER)]:
    resp = client.messages.create(
        model=MODEL, max_tokens=400, tools=[tool],
        messages=[{"role": "user", "content": PROMPT}],
    )
    print(f"--- {label} description ---")
    print(f"stop_reason: {resp.stop_reason}")
    for block in resp.content:
        if block.type == "text":
            print(f"text: {block.text[:200]}")
        elif block.type == "tool_use":
            print(f"tool_use: {block.name}({dict(block.input)})")
    print()


## All four `tool_choice` modes

Run the same prompt against each mode. Notice how `stop_reason` flips between `tool_use` and `end_turn` based purely on the mode.


In [ ]:
WEATHER_TOOL = OPINIONATED_WEATHER  # reuse from the previous cell
USER = "I'm planning a picnic in Boston tomorrow. Should I go?"

modes = [
    {"type": "auto"},
    {"type": "any"},
    {"type": "tool", "name": "get_weather"},
    {"type": "none"},
]

for mode in modes:
    resp = client.messages.create(
        model=MODEL, max_tokens=300, tools=[WEATHER_TOOL], tool_choice=mode,
        messages=[{"role": "user", "content": USER}],
    )
    print(f"mode={mode} -> stop_reason={resp.stop_reason}")
    for block in resp.content:
        if block.type == "tool_use":
            print(f"  tool_use: {block.name}({dict(block.input)})")
        elif block.type == "text":
            print(f"  text: {block.text[:120]!r}")
    print()


## Structured error contract

A helper to build the structured error payload. We show two cases: a **transient** error (the model should retry) and a **policy** error (the model must not).


In [ ]:
def make_tool_error(category: str, message: str, *, retryable: bool) -> dict:
    """Build a structured tool-result error payload.

    The model reads errorCategory and isRetryable to decide whether to
    retry, reformulate, or escalate. Bare strings force it to guess.
    """
    assert category in {"transient", "permanent", "policy"}, category
    return {
        "isError": True,
        "errorCategory": category,
        "isRetryable": retryable,
        "message": message,
    }


print("transient (retryable):")
print(json.dumps(make_tool_error("transient",
                                 "weather API timed out", retryable=True), indent=2))
print()
print("policy (not retryable):")
print(json.dumps(make_tool_error("policy",
                                 "weather API blocked for this region", retryable=False),
                 indent=2))


## `.mcp.json` as config-as-data

Open `../.mcp.json` and pretty-print its structure. Four servers, three transports, env-var expansion in the right places. This is the file the cohort can copy and modify for their own projects.


In [ ]:
mcp_path = REPO_ROOT / ".mcp.json"
config = json.loads(mcp_path.read_text(encoding="utf-8"))
servers = config["mcpServers"]

print(f"{len(servers)} servers configured\n")
for name, spec in servers.items():
    if "command" in spec:
        transport = "stdio"
        endpoint = f"{spec['command']} {' '.join(spec.get('args', []))}"
    elif spec.get("type") == "sse":
        transport = "SSE"
        endpoint = spec.get("url", "")
    elif spec.get("type") == "http":
        transport = "HTTP"
        endpoint = spec.get("url", "")
    else:
        transport = "unknown"
        endpoint = ""

    # Find env-var expansion points
    raw = json.dumps(spec)
    env_refs = sorted(set(s for s in raw.split("${") if "}" in s for s in [s.split("}", 1)[0]] if s))

    print(f"  {name}")
    print(f"    transport: {transport}")
    print(f"    endpoint:  {endpoint[:80]}")
    if env_refs:
        print(f"    env vars:  {env_refs}")
    print()


## CLAUDE.md hierarchy walk

We will not call the API here - we just print what each tier covers in this repo.


In [ ]:
tiers = [
    ("User", Path.home() / ".claude" / "CLAUDE.md", "personal defaults, every project"),
    ("Project", REPO_ROOT / "CLAUDE.md", "team conventions, checked in"),
    ("Subtree", REPO_ROOT / "notebooks" / "CLAUDE.md", "loaded on demand"),
    ("Local override", REPO_ROOT / "CLAUDE.local.md", "gitignored, personal tweaks"),
]
for tier, path, purpose in tiers:
    present = "[present]" if path.exists() else "[absent ]"
    print(f"{present} {tier:<16} {path}")
    print(f"           {purpose}\n")


## `claude -p`: the headless surface

The Claude Code CLI runs interactively *and* headlessly. `claude -p "<prompt>"` is the bridge to CI/CD.

```powershell
# audit the project CLAUDE.md
claude -p "audit ./CLAUDE.md against repo conventions. List 3 specific improvements."

# JSON output for scripting
claude -p "list all Python files with missing docstrings" --output-format json
```

Pipe the JSON into a GitHub Actions step and you have an LLM-backed lint that runs on every PR. No notebook cell here because invoking the Claude Code CLI from inside a Jupyter kernel cross-talks with the SDK auth surface. Run those commands in your PowerShell terminal between segments.


## Exercise (5 minutes)

Sketch a **CLAUDE.md hierarchy for a 3-team monorepo** (backend / frontend / infra). Name **three files**:

1. One **project-root** `CLAUDE.md`
2. Two **subtree** `CLAUDE.md` files

For each, write a **one-line statement** of what belongs there and what does **not**.

Deliverable: three file paths plus three one-liners in chat.


## Key takeaways

- **Tool descriptions are the contract**, names are just labels. Spell out behavior, inputs, error shapes, and when *not* to call.
- The four **`tool_choice`** modes are `auto`, `any`, `tool`, `none`. `tool` is the lever for forced structured output (Segment 3).
- **Structured errors** with `errorCategory` and `isRetryable` let the model decide. Bare strings force it to guess.
- **MCP transports** are stdio / SSE / HTTP, and `${ENV_VAR}` expansion keeps secrets out of source.
- **CLAUDE.md hierarchy** layers from user to project to subtree to local. Use subtree files to keep frontend rules off backend files.
- `claude -p` is your CI/CD answer.


## Bridge to Segment 3

> "Tools give the agent hands. Prompts and schemas decide what comes out. Let's make those outputs trustworthy."

Open `segment-3-invoice-extractor.ipynb`.
